In [1]:

# %%
# ==============================================
# Powercast — Week 3 Section 2 (Unified, Single Cell)
# MLflow Experiment Tracking & Evaluation (Business)
# - PROFILE support: dev | preprod | final
# - Model toggles: baseline, sarimax, prophet, xgb
# - Frequency-aware; dev-safe; progress prints
# - Writes to <BASE_DIR>/results/Wk03_Section2_<profile>/{reports,plots,mlruns}
# - Upserts Section 2 block into consolidated Week 3 report (no duplicates)
# ==============================================

# ---------- Top-level configuration ----------
PROFILE = "dev"        # dev | preprod | final
MODELS  = {            # toggle models here
    "baseline": True,
    "sarimax":  True,   # safe SARIMAX (hourly in dev/preprod), native in final
    "prophet":  False,  # requires prophet/cmdstanpy; hourly in dev
    "xgb":      False,  # requires xgboost; recursive forecast
}
SARIMAX_MODE = {       # how SARIMAX fits per profile
    "dev":     "hourly",
    "preprod": "hourly",
    "final":   "native",
}

# ---------- Safety & stability ----------
from pathlib import Path
import os, re, sys, time, math, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

# Headless plotting
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Cap threads to avoid BLAS/OMP storms
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Optional libraries
_HAVE_SM = False
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    _HAVE_SM = True
except Exception:
    _HAVE_SM = False

_HAVE_FBPROPHET = False
try:
    try:
        from prophet import Prophet
        _HAVE_FBPROPHET = True
    except Exception:
        from fbprophet import Prophet  # older fallback
        _HAVE_FBPROPHET = True
except Exception:
    _HAVE_FBPROPHET = False

_HAVE_XGB = False
try:
    from xgboost import XGBRegressor as _XGB
    def XGBRegressor(*args, **kwargs):
        kwargs.setdefault("n_estimators", 200 if PROFILE=="dev" else 400)
        kwargs.setdefault("max_depth", 4)
        kwargs.setdefault("learning_rate", 0.1)
        kwargs.setdefault("subsample", 0.9)
        kwargs.setdefault("colsample_bytree", 0.9)
        kwargs.setdefault("tree_method", "hist")
        kwargs.setdefault("n_jobs", 1)
        return _XGB(*args, **kwargs)
    _HAVE_XGB = True
except Exception:
    _HAVE_XGB = False

_HAVE_MLFLOW = False
try:
    import mlflow
    _HAVE_MLFLOW = True
except Exception:
    _HAVE_MLFLOW = False

# Silence statsmodels frequency message; we manage freq explicitly
warnings.filterwarnings("ignore", message=r"No frequency information was provided, so inferred frequency .* will be used\.")

# ---------- Paths & profile envelope ----------
BASE_DIR = Path.cwd()
if BASE_DIR.name.lower() == "code":
    BASE_DIR = BASE_DIR.parent

SECTION_ID = "Wk03_Section2"
CFG = {
    "dev":     dict(max_days=90,   zones=1, test_days=7),
    "preprod": dict(max_days=365,  zones=3, test_days=28),
    "final":   dict(max_days=None, zones=3, test_days=28),
}
cfg = CFG.get(PROFILE, CFG["dev"])

RESULTS_DIR = BASE_DIR / "results" / f"{SECTION_ID}_{PROFILE}"
REPORTS_DIR = RESULTS_DIR / "reports"
PLOTS_DIR   = RESULTS_DIR / "plots"
MLRUNS_DIR  = RESULTS_DIR / "mlruns"
for d in (REPORTS_DIR, PLOTS_DIR, MLRUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

DATA_PATH    = BASE_DIR / "data" / "Tetuan City power consumption.csv"
CONSOLIDATED = BASE_DIR / "SDS-CP036-powercast_Wk03_Report_Business.md"

def rel(p: Path) -> str:
    try: return p.relative_to(BASE_DIR).as_posix()
    except Exception: return p.as_posix()

print(f"[BOOT] BASE_DIR={BASE_DIR}")
print(f"[CFG ] PROFILE={PROFILE} zones={cfg['zones']} max_days={cfg['max_days']} test_days={cfg['test_days']}")
print(f"[MODELS] baseline={MODELS.get('baseline')}  sarimax={MODELS.get('sarimax')} (avail={_HAVE_SM})  prophet={MODELS.get('prophet')} (avail={_HAVE_FBPROPHET})  xgb={MODELS.get('xgb')} (avail={_HAVE_XGB})")
print(f"[MLflow] available={_HAVE_MLFLOW}")
print(f"[OUT ] {RESULTS_DIR}")

# ---------- Helpers ----------
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"\s+", " ", c.strip()) for c in df.columns]
    return df

def infer_step_and_freq(idx: pd.DatetimeIndex):
    try:
        freq = pd.infer_freq(idx[:1000])
    except Exception:
        freq = None
    if freq:
        step = pd.to_timedelta(pd.tseries.frequencies.to_offset(freq))
        return step, freq
    diffs = pd.Series(idx).diff().dropna()
    step = diffs.mode().iloc[0] if not diffs.empty else pd.Timedelta(hours=1)
    total_sec = int(step.total_seconds())
    if total_sec % 3600 == 0:
        freq = f"{total_sec // 3600}H"
    elif total_sec % 60 == 0:
        freq = f"{total_sec // 60}T"
    else:
        freq = f"{total_sec}S"
    return step, freq

def limit_last_days(df: pd.DataFrame, date_col="DateTime", max_days=None) -> pd.DataFrame:
    if not max_days:
        return df
    df = df.copy()
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.sort_values(date_col)
        cutoff = df[date_col].max() - pd.Timedelta(days=max_days)
        df = df[df[date_col] >= cutoff]
    return df

def pick_zone_cols(df: pd.DataFrame, k: int):
    cols = [c for c in df.columns if re.match(r"Zone\s+\d+\s+Power\s+Consumption", c)]
    cols = sorted(cols, key=lambda s: int(re.search(r"\d+", s).group()))
    return cols[:k] if k else cols

def make_time_splits(df: pd.DataFrame, test_days: int, date_col="DateTime", steps_per_day: int = 24):
    df = df.sort_values(date_col)
    test_len = int(test_days * steps_per_day)
    if test_len >= len(df):
        test_len = max(1, len(df)//10)  # safeguard
    test = df.iloc[-test_len:].copy()
    train_val = df.iloc[:-test_len].copy()
    if len(train_val) > 100:
        cut = int(len(train_val) * 0.85)
        train, val = train_val.iloc[:cut].copy(), train_val.iloc[cut:].copy()
    else:
        train, val = train_val.copy(), pd.DataFrame(columns=train_val.columns)
    return train, val, test

def metrics_series(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    denom = np.maximum(1e-8, np.where(y_true==0, 1e-8, y_true))
    mape = float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)
    return dict(rmse=rmse, mae=mae, mape=mape)

def save_plot_ts(ts_df: pd.DataFrame, cols, title, path: Path):
    plt.figure(figsize=(10,4))
    for c in cols:
        plt.plot(ts_df.index, ts_df[c], label=c)
    plt.title(title)
    plt.xlabel("Time"); plt.ylabel("Power")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close("all")

# ---------- Load & prepare ----------
if not DATA_PATH.exists():
    msg = f"Data file not found: {rel(DATA_PATH)}"
    print("[ERROR]", msg)
    # minimal section + consolidated for pipelines
    (REPORTS_DIR / "SDS-CP036-powercast_Wk03_Section2_Business_Report.md").write_text(
        f"# SDS-CP036-powercast - Week 3 Section 2: MLflow Experiment Tracking & Evaluation\n\n"
        f"Profile: `{PROFILE}`\n\n"
        f"**Error:** {msg}\n", encoding="utf-8"
    )
    raise SystemExit(0)

df = pd.read_csv(DATA_PATH)
df = normalize_columns(df)

if "DateTime" not in df.columns:
    alt = [c for c in df.columns if c.lower() == "datetime"]
    if alt:
        df = df.rename(columns={alt[0]: "DateTime"})
    else:
        raise RuntimeError("Missing DateTime column. Got: " + ", ".join(df.columns))

df = limit_last_days(df, "DateTime", cfg["max_days"])
zones = pick_zone_cols(df, cfg["zones"])
if not zones:
    raise RuntimeError("No 'Zone X Power Consumption' columns found. Got: " + ", ".join(df.columns))

work = df[["DateTime"] + zones].copy()
work["DateTime"] = pd.to_datetime(work["DateTime"])
work = work.sort_values("DateTime").set_index("DateTime")

step_td, freq_str = infer_step_and_freq(work.index)
steps_per_hour = max(1, int(round(pd.Timedelta(hours=1) / step_td)))
steps_per_day  = int(24 * steps_per_hour)
print(f"[FREQ] step={step_td} freq='{freq_str}' steps/hour={steps_per_hour} steps/day={steps_per_day}")

# ---------- MLflow setup (local file store) ----------
if _HAVE_MLFLOW:
    tracking_uri = "file://" + str(MLRUNS_DIR.resolve())
    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(f"Powercast_Wk03_S2_{PROFILE}")
    print(f"[MLflow] tracking_uri={tracking_uri}")
else:
    print("[MLflow] package not available; will emit CSV-only tracking.")

# ---------- Model runners ----------
def run_baseline(train, val, test, zone):
    last_val = (val[zone].iloc[-1] if len(val) else train[zone].iloc[-1])
    y_pred = np.repeat(last_val, len(test))
    return y_pred, {"baseline_last": float(last_val)}

def run_sarimax(series_native, test_index, steps_per_hour, mode="hourly"):
    if not _HAVE_SM:
        raise RuntimeError("statsmodels not available")
    series_native = pd.to_numeric(series_native, errors="coerce").ffill()
    params = {}
    if mode == "hourly":
        series_hr = series_native.resample("1H").mean().asfreq("H").ffill()
        s = 24
        if len(series_hr) > 2400:
            series_hr = series_hr.iloc[-2400:]
        params.update({"order":"(1,0,1)","seasonal":"(0,1,1,24)","mode":"hourly"})
        model = SARIMAX(series_hr, order=(1,0,1), seasonal_order=(0,1,1,s),
                        enforce_stationarity=False, enforce_invertibility=False,
                        simple_differencing=True, trend=None, concentrate_scale=True)
        res = model.fit(method="lbfgs", maxiter=40, disp=False)
        h = int(math.ceil(len(test_index) / steps_per_hour))
        fc_hr = res.get_forecast(steps=h)
        yhat_hr = pd.Series(fc_hr.predicted_mean.values.astype(float),
                            index=pd.date_range(series_hr.index[-1] + pd.Timedelta(hours=1), periods=h, freq="H"))
        native_hours = pd.to_datetime(pd.Index(test_index).floor("H"))
        yhat_native = yhat_hr.reindex(native_hours, method="ffill").values[:len(test_index)]
        return yhat_native, params
    else:
        s = steps_per_day
        series_nat = series_native.asfreq(freq_str).ffill()
        maxlen = 144*200 if steps_per_day>=144 else 24*400
        if len(series_nat) > maxlen:
            series_nat = series_nat.iloc[-maxlen:]
        params.update({"order":"(1,0,1)","seasonal":f"(0,1,1,{s})","mode":"native"})
        model = SARIMAX(series_nat, order=(1,0,1), seasonal_order=(0,1,1,s),
                        enforce_stationarity=False, enforce_invertibility=False,
                        simple_differencing=True, trend=None, concentrate_scale=True)
        res = model.fit(method="lbfgs", maxiter=60 if PROFILE!="dev" else 40, disp=False)
        fc = res.get_forecast(steps=len(test_index))
        return fc.predicted_mean.values.astype(float), params

def run_prophet(series_native, test_len, steps_per_hour):
    if not _HAVE_FBPROPHET:
        raise RuntimeError("prophet not available")
    series_hr = pd.to_numeric(series_native, errors="coerce").ffill().resample("1H").mean().asfreq("H").ffill()
    dfp = pd.DataFrame({"ds": series_hr.index, "y": series_hr.values.astype(float)})
    m = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False,
                changepoint_prior_scale=0.2)
    m.fit(dfp)
    h = int(math.ceil(test_len / steps_per_hour))
    future = m.make_future_dataframe(periods=h, freq="H", include_history=False)
    fc = m.predict(future)
    yhat_hr = pd.Series(fc["yhat"].values.astype(float), index=future["ds"].values)
    return yhat_hr, {"mode":"hourly","daily_seasonality":True,"weekly_seasonality":True}

def run_xgb_recursive(series_native, test_index, max_lag=24):
    if not _HAVE_XGB:
        raise RuntimeError("xgboost not available")
    y = pd.to_numeric(series_native, errors="coerce").ffill()
    df_sup = pd.DataFrame({"y": y})
    for L in range(1, max_lag+1):
        df_sup[f"lag_{L}"] = y.shift(L)
    df_sup = df_sup.dropna()
    max_rows = 5000 if PROFILE=="dev" else 20000
    if len(df_sup) > max_rows:
        df_sup = df_sup.iloc[-max_rows:]
    X_train = df_sup.drop(columns=["y"]).values
    y_train = df_sup["y"].values
    model = XGBRegressor()
    model.fit(X_train, y_train, verbose=False)
    history = list(y.values)
    yhat = []
    for ts in test_index:
        feats = [history[-k] for k in range(1, max_lag+1)]
        pred = float(model.predict(np.array(feats).reshape(1,-1)))
        yhat.append(pred)
        history.append(pred)
    return np.array(yhat, dtype=float), {"lags":max_lag,"n_rows":int(len(df_sup))}

# ---------- Split ----------
train, val, test = make_time_splits(work.reset_index(), cfg["test_days"], "DateTime", steps_per_day=steps_per_day)
train = train.set_index("DateTime"); val = val.set_index("DateTime"); test = test.set_index("DateTime")

# ---------- Experiment loop ----------
rows = []
for zone in zones:
    print(f"[ZONE] {zone}")
    y_true = test[zone].values
    series_native = pd.concat([train[zone], val[zone]]) if len(val) else train[zone]

    # Baseline
    if MODELS.get("baseline", True):
        y_pred, params = run_baseline(train, val, test, zone)
        m = metrics_series(y_true, y_pred)
        model_name = "Baseline (Naïve)"
        rows.append({"zone":zone, "model":model_name, **m, "profile":PROFILE})
        # plot
        comp = pd.DataFrame({"actual": test[zone], "pred": y_pred}, index=test.index)
        pth = PLOTS_DIR / f"{zone.replace(' ','_')}_actual_vs_baseline.png"
        save_plot_ts(comp, ["actual","pred"], f"{zone} — Actual vs Baseline", pth)
        # mlflow
        if _HAVE_MLFLOW:
            with mlflow.start_run(run_name=f"{zone}__Baseline"):
                mlflow.log_params({"model":"baseline", "profile":PROFILE, **params})
                for k,v in m.items(): mlflow.log_metric(k, float(v))
                mlflow.log_artifact(str(pth))

    # SARIMAX
    if MODELS.get("sarimax", False) and _HAVE_SM:
        mode = SARIMAX_MODE.get(PROFILE, "hourly")
        try:
            y_pred, params = run_sarimax(series_native, test.index, steps_per_hour, mode=mode)
            m = metrics_series(y_true, y_pred)
            model_name = f"SARIMAX@{mode}"
            rows.append({"zone":zone, "model":model_name, **m, "profile":PROFILE})
            comp = pd.DataFrame({"actual": test[zone], "pred": y_pred}, index=test.index)
            pth = PLOTS_DIR / f"{zone.replace(' ','_')}_actual_vs_sarimax.png"
            save_plot_ts(comp, ["actual","pred"], f"{zone} — Actual vs SARIMAX ({mode})", pth)
            if _HAVE_MLFLOW:
                with mlflow.start_run(run_name=f"{zone}__SARIMAX_{mode}"):
                    mlflow.log_params({"model":"sarimax", "profile":PROFILE, **params})
                    for k,v in m.items(): mlflow.log_metric(k, float(v))
                    mlflow.log_artifact(str(pth))
        except Exception as e:
            print(f"  [WARN] SARIMAX failed: {e}")

    # Prophet
    if MODELS.get("prophet", False) and _HAVE_FBPROPHET:
        try:
            y_hr, params = run_prophet(series_native, len(test), steps_per_hour)
            native_hours = pd.to_datetime(test.index.floor("H"))
            y_pred = pd.Series(y_hr.values, index=pd.to_datetime(y_hr.index)).reindex(native_hours, method="ffill").values[:len(test)]
            m = metrics_series(y_true, y_pred)
            model_name = "Prophet@1H->native"
            rows.append({"zone":zone, "model":model_name, **m, "profile":PROFILE})
            comp = pd.DataFrame({"actual": test[zone], "pred": y_pred}, index=test.index)
            pth = PLOTS_DIR / f"{zone.replace(' ','_')}_actual_vs_prophet.png"
            save_plot_ts(comp, ["actual","pred"], f"{zone} — Actual vs Prophet (1H)", pth)
            if _HAVE_MLFLOW:
                with mlflow.start_run(run_name=f"{zone}__Prophet"):
                    mlflow.log_params({"model":"prophet", "profile":PROFILE, **params})
                    for k,v in m.items(): mlflow.log_metric(k, float(v))
                    mlflow.log_artifact(str(pth))
        except Exception as e:
            print(f"  [WARN] Prophet failed: {e}")
    elif MODELS.get("prophet", False) and not _HAVE_FBPROPHET:
        print("  [SKIP] Prophet selected but package not available.")

    # XGBoost
    if MODELS.get("xgb", False) and _HAVE_XGB:
        try:
            y_pred, params = run_xgb_recursive(series_native, test.index, max_lag=24)
            m = metrics_series(y_true, y_pred)
            model_name = "XGBoost (lags)"
            rows.append({"zone":zone, "model":model_name, **m, "profile":PROFILE})
            comp = pd.DataFrame({"actual": test[zone], "pred": y_pred}, index=test.index)
            pth = PLOTS_DIR / f"{zone.replace(' ','_')}_actual_vs_xgb.png"
            save_plot_ts(comp, ["actual","pred"], f"{zone} — Actual vs XGBoost (lags)", pth)
            if _HAVE_MLFLOW:
                with mlflow.start_run(run_name=f"{zone}__XGB"):
                    mlflow.log_params({"model":"xgb_lags", "profile":PROFILE, **params})
                    for k,v in m.items(): mlflow.log_metric(k, float(v))
                    mlflow.log_artifact(str(pth))
        except Exception as e:
            print(f"  [WARN] XGBoost failed: {e}")
    elif MODELS.get("xgb", False) and not _HAVE_XGB:
        print("  [SKIP] XGBoost selected but package not available.")

# ---------- Emit metrics CSV ----------
metrics_csv = REPORTS_DIR / "experiment_metrics.csv"
pd.DataFrame(rows).to_csv(metrics_csv, index=False)
print(f"✅ Saved experiment metrics to: {rel(metrics_csv)}")

# ---------- Build Section 2 business report ----------
section2_md = REPORTS_DIR / "SDS-CP036-powercast_Wk03_Section2_Business_Report.md"
section2_text = f"""# SDS-CP036-powercast – Week 3 Section 2: MLflow Experiment Tracking & Evaluation

Profile: `{PROFILE}`

### Key Questions Answered

**Q: Which evaluation metrics did you use to assess model performance, and why are they appropriate for this problem?**  
**A:** We report **RMSE** (penalizes bigger errors), **MAE** (average absolute miss in original units), and **MAPE** (percent error, easy to read for business). Together they show both **size of mistakes** and **relative impact**.

**Q: How did you use MLflow (or another tool) to track your experiments and results?**  
**A:** When MLflow is available, we:
- Create a local tracking store under `{rel(MLRUNS_DIR)}` and an experiment named `Powercast_Wk03_S2_{PROFILE}`.
- Log **parameters** (model type, seasonal settings, lags) and **metrics** (RMSE/MAE/MAPE) **per zone**.
- Attach **artifacts** (actual vs predicted plots) to each run for quick visual review.  
If MLflow isn’t available, we still write a consolidated CSV: `{rel(metrics_csv)}`.

**Q: What insights did you gain from comparing actual vs. predicted curves for each zone?**  
**A:** The plots in `{rel(PLOTS_DIR)}/` make **pattern fit** transparent:
- If the model tracks the **daily peaks and troughs**, it’s capturing seasonality.
- If forecasts **lag changes** or **flatten peaks**, we switch/tune models (e.g., enable XGBoost for non‑linear spikes, use native‑freq SARIMAX in `final`).

### What’s included in this run
- Zones modeled: {cfg['zones']}  
- Data window: Last {cfg['max_days'] or 'all available'} days  
- Test horizon: {cfg['test_days']} day(s) × native steps/day = {cfg['test_days']*steps_per_day} steps

**Artifacts for reviewers**
- Metrics table: `{rel(metrics_csv)}`  
- MLflow runs (if enabled): `{rel(MLRUNS_DIR)}`  
- Visuals: `{rel(PLOTS_DIR)}/`

### Business Value (Executive View)
- **Traceability:** Every run is documented with parameters, metrics, and plots.  
- **Faster decisions:** Side‑by‑side runs quickly identify the **champion** model per zone.  
- **Governance‑ready:** Local MLflow store + CSV provide an audit trail suitable for leadership reviews.
"""

section2_md.write_text(section2_text, encoding="utf-8")
print(f"✅ Wrote section report: {rel(section2_md)}")

# ---------- Consolidated upsert (replace-or-append Section 2) ----------
def upsert_section_2(consolidated_path: Path, new_block: str):
    # Ensure a base file exists
    if not consolidated_path.exists():
        base = [
            "# SDS-CP036-powercast – Week 3 (Business Report)",
            "",
            "## Table of Contents",
            "- [Section 1 – Model Selection & Training](#sds-cp036-powercast-–-week-3-section-1-model-selection--training)",
            "- [Section 2 – MLflow Experiment Tracking & Evaluation](#sds-cp036-powercast-–-week-3-section-2-mlflow-experiment-tracking--evaluation)",
            "- [Section 3 – Evaluation & Model Interpretation & Insights](#sds-cp036-powercast-–-week-3-section-3-evaluation--interpretation--insights)",
            ""
        ]
        consolidated_path.write_text("\n".join(base), encoding="utf-8")

    text = consolidated_path.read_text(encoding="utf-8")

    # Patterns
    hdr_s1 = re.compile(r"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*3\s*Section\s*1\s*:\s*Model\s*Selection\s*\&\s*Training\s*$")
    hdr_s2 = re.compile(r"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*3\s*Section\s*2\s*:\s*MLflow\s*Experiment\s*Tracking\s*\&\s*Evaluation\s*$")
    hdr_any = re.compile(r"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*3\s*Section\s*[1-3]\s*:\s*.*$")

    # Remove any existing Section 2 block
    def remove_block(txt, hdr_regex):
        m = hdr_regex.search(txt)
        if not m:
            return txt
        start = m.start()
        m2 = hdr_any.search(txt, pos=m.end())
        end = m2.start() if m2 else len(txt)
        return (txt[:start] + txt[end:]).strip()

    text = remove_block(text, hdr_s2).strip()

    # Insert after Section 1 if present, else append at end
    m1 = hdr_s1.search(text)
    if m1:
        # Find end of Section 1 block
        mnext = hdr_any.search(text, pos=m1.end())
        end1 = mnext.start() if mnext else len(text)
        new_text = text[:end1].rstrip() + "\n\n" + new_block.strip() + "\n\n" + text[end1:].lstrip()
    else:
        new_text = text.rstrip() + "\n\n" + new_block.strip() + "\n"

    consolidated_path.write_text(new_text, encoding="utf-8")
    print(f"✅ Upserted Section 2 into consolidated: {rel(consolidated_path)}")

# Build a clean Section 2 block to upsert
s2_block = section2_md.read_text(encoding="utf-8").strip()
upsert_section_2(CONSOLIDATED, s2_block)

# ---------- Tail print ----------
try:
    disp = pd.read_csv(metrics_csv)
    print("\n--- Metrics (head) ---")
    print(disp.head(10).to_string(index=False))
except Exception as e:
    print("Unable to show metrics head:", e)
print("\nDone.")


[BOOT] BASE_DIR=/home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast
[CFG ] PROFILE=dev zones=1 max_days=90 test_days=7
[MODELS] baseline=True  sarimax=True (avail=True)  prophet=False (avail=True)  xgb=False (avail=True)
[MLflow] available=False
[OUT ] /home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast/results/Wk03_Section2_dev
[FREQ] step=0 days 00:10:00 freq='10T' steps/hour=6 steps/day=144
[MLflow] package not available; will emit CSV-only tracking.
[ZONE] Zone 1 Power Consumption
✅ Saved experiment metrics to: results/Wk03_Section2_dev/reports/experiment_metrics.csv
✅ Wrote section report: results/Wk03_Section2_dev/reports/SDS-CP036-powercast_Wk03_Section2_Business_Report.md
✅ Upserted Section 2 into consolidated: SDS-CP036-powercast_Wk03_Report_Business.md

--- Metrics (head) ---
                    zone            model         rmse          mae      mape profile
Zone 1 Power Consumption Baseline (Naïve)  6357.359740  5470.414630 19.382757     dev
Zone 1 Power Consumption   SA